## Installation of LLama3

In [1]:
!apt-get update
!apt-get install -y zstd


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,615 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!nohup ollama serve > ollama.log 2>&1 &


In [4]:
import time
time.sleep(5)
!which ollama


/usr/local/bin/ollama


#### Saving llama3 locally temporarily

In [5]:

!pip install ollama
!ollama pull llama3



## Getting user description

In [86]:
problem_text = ("""31. Next Permutation
Attempted
Medium
Topics
premium lock icon
Companies
A permutation of an array of integers is an arrangement of its members into a sequence or linear order.

For example, for arr = [1,2,3], the following are all the permutations of arr: [1,2,3], [1,3,2], [2, 1, 3], [2, 3, 1], [3,1,2], [3,2,1].
The next permutation of an array of integers is the next lexicographically greater permutation of its integer. More formally, if all the permutations of the array are sorted in one container according to their lexicographical order, then the next permutation of that array is the permutation that follows it in the sorted container. If such arrangement is not possible, the array must be rearranged as the lowest possible order (i.e., sorted in ascending order).

For example, the next permutation of arr = [1,2,3] is [1,3,2].
Similarly, the next permutation of arr = [2,3,1] is [3,1,2].
While the next permutation of arr = [3,2,1] is [1,2,3] because [3,2,1] does not have a lexicographical larger rearrangement.
Given an array of integers nums, find the next permutation of nums.

The replacement must be in place and use only constant extra memory.



Example 1:

Input: nums = [1,2,3]
Output: [1,3,2]
Example 2:

Input: nums = [3,2,1]
Output: [1,2,3]
Example 3:

Input: nums = [1,1,5]
Output: [1,5,1]


Constraints:

1 <= nums.length <= 100
0 <= nums[i] <= 100

class Solution:
    def nextPermutation(self, nums: List[int]) -> None:
        """)


In [97]:
problem_text=("""34. Find First and Last Position of Element in Sorted Array
Attempted
Medium
Topics
premium lock icon
Companies
Given an array of integers nums sorted in non-decreasing order, find the starting and ending position of a given target value.

If target is not found in the array, return [-1, -1].

You must write an algorithm with O(log n) runtime complexity.



Example 1:

Input: nums = [5,7,7,8,8,10], target = 8
Output: [3,4]
Example 2:

Input: nums = [5,7,7,8,8,10], target = 6
Output: [-1,-1]
Example 3:

Input: nums = [], target = 0
Output: [-1,-1]


Constraints:

0 <= nums.length <= 105
-109 <= nums[i] <= 109
nums is a non-decreasing array.
-10
class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
      """)

In [107]:
problem_text= ("""53. Maximum Subarray
Medium
Topics
premium lock icon
Companies
Given an integer array nums, find the subarray with the largest sum, and return its sum.



Example 1:

Input: nums = [-2,1,-3,4,-1,2,1,-5,4]
Output: 6
Explanation: The subarray [4,-1,2,1] has the largest sum 6.
Example 2:

Input: nums = [1]
Output: 1
Explanation: The subarray [1] has the largest sum 1.
Example 3:

Input: nums = [5,4,-1,7,8]
Output: 23
Explanation: The subarray [5,4,-1,7,8] has the largest sum 23.


Constraints:

1 <= nums.length <= 105
-104 <= nums[i] <= 104

solution: class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
 """)

## Preprocessing the input to extract problem description, constraints, solution if any

In [108]:
import ollama
import json
import re

MODEL_NAME = "llama3"


# =========================
# JSON UTILS
# =========================
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else ""


def safe_parse(text):
    try:
        return json.loads(text)
    except:
        return None


# =========================
# DESCRIPTION PASS (LLM)
# =========================
def parse_description(text):
    prompt = f"""
Extract the problem description from the input.

Return STRICT JSON:
{{ "description": "" }}

RULES:
- ONLY JSON output
- DO NOT hallucinate or add new facts
- Extract ONLY from INPUT
- You MAY clean formatting noise like:
  - "Solved", "Medium", "Topics", etc.
- Preserve:
  - all problem statements
  - all examples
- EXCLUDE:
  - constraints section
  - code

CRITICAL:
- If unsure, still extract best possible description
- Never return null or empty unless input is empty

INPUT START
{text}
INPUT END
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 600}
    )

    parsed = safe_parse(extract_json(res["message"]["content"]))
    return parsed["description"] if parsed and "description" in parsed else ""

# =========================
# CONSTRAINTS PASS (LLM)
# =========================
def parse_constraints(text):
    prompt = f"""
Extract OR infer constraints.

Return STRICT JSON:
{{ "constraints": "" }}

RULES:
- If constraints exist → copy EXACTLY
- If missing → infer realistic constraints
- Format:
  1 <= n <= 10^5
  0 <= arr[i] <= 10^4
- No explanations

-------------------------
FEW SHOTS
-------------------------

INPUT:
Constraints:
1 <= strs.length <= 200
0 <= strs[i].length <= 200

OUTPUT:
{{ "constraints": "1 <= strs.length <= 200\\n0 <= strs[i].length <= 200" }}

-------------------------

INPUT:
Constraints:
3 <= nums.length <= 3000
-10^5 <= nums[i] <= 10^5

OUTPUT:
{{ "constraints": "3 <= nums.length <= 3000\\n-10^5 <= nums[i] <= 10^5" }}

-------------------------

INPUT:
3Sum Closest

OUTPUT:
{{ "constraints": "3 <= nums.length <= 500\\n-1000 <= nums[i] <= 1000\\n-10^4 <= target <= 10^4" }}

-------------------------
INPUT:
{text}
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "num_predict": 150}
    )

    parsed = safe_parse(extract_json(res["message"]["content"]))
    return parsed["constraints"] if parsed else ""


# =========================
# SOLUTION PASS (REGEX — RELIABLE)
# =========================
def parse_solution(text):
    """
    Extract FIRST class Solution block reliably
    """

    pattern = r"class Solution:\n(?:\s+.*\n?)*"
    match = re.search(pattern, text)

    if match:
        return match.group(0).rstrip()

    return ""


# =========================
# MAIN PIPELINE
# =========================
def structured_preprocess(text):

    description = parse_description(text)
    constraints = parse_constraints(text)
    solution = parse_solution(text)   # <-- DO NOT JSON PARSE

    return {
        "description": description,
        "constraints": constraints,
        "solution": solution
    }


# =========================
# RUN
# =========================
structured_data = structured_preprocess(problem_text)
print(structured_data)


{'description': 'Given an integer array nums, find the subarray with the largest sum, and return its sum.', 'constraints': '1 <= nums.length <= 10^5\n-10^4 <= nums[i] <= 10^4', 'solution': 'class Solution:\n    def maxSubArray(self, nums: List[int]) -> int:'}


In [109]:
print("description:\n", structured_data["description"])
description = structured_data["description"]
print("\nconstraints:\n", structured_data["constraints"])
constraints= structured_data["constraints"]
print("\nsolution:\n", structured_data["solution"])
solution = structured_data["solution"]


description:
 Given an integer array nums, find the subarray with the largest sum, and return its sum.

constraints:
 1 <= nums.length <= 10^5
-10^4 <= nums[i] <= 10^4

solution:
 class Solution:
    def maxSubArray(self, nums: List[int]) -> int:


## getting sample test cases

In [110]:
import re
import ast


# =========================
# SAMPLE TEST EXTRACTION (FIXED)
# =========================
def extract_samples(text):
    """
    Extracts (input, output) pairs from coding problem text.
    Properly stops at Explanation to avoid leakage.
    """

    text = text.replace("\r", "")

    # ✅ FIX: stop at Explanation as well
    pattern = r"Input:\s*(.*?)\n\s*Output:\s*(.*?)(?=\n\s*(Explanation:|Input:|Example|Constraints|$))"

    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)

    samples = []
    for inp, out, _ in matches:
        samples.append({
            "input": inp.strip(),
            "output": out.strip()
        })

    return samples


# =========================
# INPUT PARSER (GENERIC ENOUGH FOR NOW)
# =========================
def parse_input_string(input_str):
    """
    Keeps raw structure + lightweight extraction.
    (Main execution still uses exec-based parser elsewhere)
    """

    arrays = re.findall(r"\[.*?\]", input_str)
    nums = re.findall(r"-?\d+", input_str)

    return {
        "arrays": arrays,
        "numbers": list(map(int, nums)) if nums else []
    }


# =========================
# OUTPUT PARSER (SIMPLIFIED + ROBUST)
# =========================
def parse_output(output_str):
    """
    Clean parser assuming extraction is correct.
    """

    output_str = output_str.strip()

    # ✅ try exact structured parse
    try:
        return ast.literal_eval(output_str)
    except:
        pass

    # ✅ fallback: single number
    nums = re.findall(r"-?\d+", output_str)
    if len(nums) == 1:
        return int(nums[0])

    # fallback raw
    return output_str


# =========================
# PIPELINE WRAPPER
# =========================
def build_testcases(problem_text):
    """
    End-to-end sample extraction pipeline.
    """

    raw_samples = extract_samples(problem_text)

    testcases = []

    for s in raw_samples:
        inp = parse_input_string(s["input"])
        out = parse_output(s["output"])

        testcases.append({
            "input_raw": s["input"],
            "input_parsed": inp,
            "expected": out
        })

    return testcases


# =========================
# EXAMPLE USAGE
# =========================
tests = build_testcases(problem_text)

print(tests)

[{'input_raw': 'nums = [-2,1,-3,4,-1,2,1,-5,4]', 'input_parsed': {'arrays': ['[-2,1,-3,4,-1,2,1,-5,4]'], 'numbers': [-2, 1, -3, 4, -1, 2, 1, -5, 4]}, 'expected': 6}, {'input_raw': 'nums = [1]', 'input_parsed': {'arrays': ['[1]'], 'numbers': [1]}, 'expected': 1}, {'input_raw': 'nums = [5,4,-1,7,8]', 'input_parsed': {'arrays': ['[5,4,-1,7,8]'], 'numbers': [5, 4, -1, 7, 8]}, 'expected': 23}]


## Creating working solution for the description and testing it on the user provided test cases.

In [111]:
import ollama
import ast
import traceback

MODEL_NAME = "llama3"


# =========================
# CLEAN CODE
# =========================
def clean_code(text):
    text = text.replace("```python", "").replace("```", "")

    idx = text.find("class Solution")
    if idx != -1:
        text = text[idx:]

    return text.strip()


# =========================
# LLM GENERATION
# =========================
def llm_generate_solution(description, constraints, solution_template, error_msg=None):
    prompt = f"""
You are a Python competitive programming engine.

STRICT RULES:
- Output ONLY executable Python code
- NO explanations
- MUST define class Solution
- MUST match function signature exactly
- ONLY complete the function body

TEMPLATE:
{solution_template}

PROBLEM:
{description}

CONSTRAINTS:
{constraints}
"""

    if error_msg:
        prompt += f"\n\nPREVIOUS ERROR:\n{error_msg}\nFix the code."

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return clean_code(res["message"]["content"])


# =========================
# PARSE INPUTS (FIXED)
# =========================
def parse_inputs(input_raw):
    env = {}

    # split ONLY on top-level commas (safe enough for LC-style inputs)
    parts = []
    current = ""
    bracket_level = 0

    for ch in input_raw:
        if ch == ',' and bracket_level == 0:
            parts.append(current.strip())
            current = ""
        else:
            current += ch
            if ch in "[{(":
                bracket_level += 1
            elif ch in "]})":
                bracket_level -= 1

    if current:
        parts.append(current.strip())

    # now execute each assignment separately
    for part in parts:
        if "=" not in part:
            continue
        exec(part, {}, env)

    return list(env.values())

# =========================
# GET FUNCTION (GENERIC)
# =========================
def get_function(sol, num_args):
    candidates = []

    for name in dir(sol):
        if name.startswith("__"):
            continue

        fn = getattr(sol, name)

        if callable(fn):
            try:
                if fn.__code__.co_argcount - 1 == num_args:
                    candidates.append((name, fn))
            except:
                continue

    if not candidates:
        raise Exception("No matching function found")

    # ✅ Prefer method that does NOT look like a helper
    # (heuristic: avoid names starting with 'find', 'helper', etc.)
    priority = []
    fallback = []

    for name, fn in candidates:
        if name.lower().startswith(("find", "helper", "dfs", "bfs", "util")):
            fallback.append(fn)
        else:
            priority.append(fn)

    if priority:
        return priority[0]

    return fallback[0]


# =========================
# RUN SINGLE TEST
# =========================
def run_once(code, test):
    try:
        env = {}

        # ensure typing import if needed
        if "List[" in code and "from typing import" not in code:
            code = "from typing import List\n" + code

        exec(code, env)

        import copy

        sol = env["Solution"]()

        args = parse_inputs(test["input_raw"])
        args_copy = copy.deepcopy(args)  # optional but useful for debugging later

        func = get_function(sol, len(args))

        result = func(*args)

        # handle in-place modification functions
        if result is None:
            if len(args) == 1:
                result = args[0]
            else:
                result = args

        return {
            "status": "OK",
            "output": result,
            "error": None,
            "code": code
        }

    except Exception:
        return {
            "status": "ERROR",
            "output": None,
            "error": traceback.format_exc(),
            "code": code
        }

# =========================
# GENERATE ONCE + TEST ALL + RETRY
# =========================
def solve_with_retry(description, constraints, solution_template, tests, max_attempts=3):
    last_error = None
    best_results = None
    best_score = -1
    best_code = None

    for _ in range(max_attempts):
        code = llm_generate_solution(description, constraints, solution_template, last_error)

        results = []
        passed = 0

        for i, t in enumerate(tests):
            res = run_once(code, t)

            is_match = (
                res["status"] == "OK" and
                res["output"] == t["expected"]
            )

            if is_match:
                passed += 1
            else:
                last_error = res["error"]

            results.append({
                "test_id": i,
                "input": t["input_raw"],
                "expected": t["expected"],
                "output": res["output"],
                "status": res["status"],
                "error": res["error"],
                "match": is_match,
                "solution_used": code
            })

        # keep best attempt
        if passed > best_score:
            best_score = passed
            best_results = results
            best_code = code

        # early stop if perfect
        if passed == len(tests):
            break

    return best_code, best_results


# =========================
# MAIN EXECUTION (YOUR FLOW)
# =========================
results = []

code, results = solve_with_retry(
    description=description,
    constraints=constraints,
    solution_template=solution,
    tests=tests,
    max_attempts=3
)


# =========================
# SUMMARY
# =========================
passed = sum(1 for r in results if r["match"])
errors = sum(1 for r in results if r["status"] == "ERROR")
total = len(results)

print(f"\nPASS RATE: {passed}/{total}")
print(f"ERRORS: {errors}/{total}\n")

for r in results:
    print("----")
    print("TEST:", r["input"])
    print("EXPECTED:", r["expected"])
    print("GOT:", r["output"])
    print("STATUS:", r["status"])
    print("MATCH:", r["match"])

    if not r["match"]:
        print("ERROR:", r["error"])

    print("\nSOLUTION USED:\n", r["solution_used"])


print("\nFINAL BEST CODE:\n")
print(code)


PASS RATE: 3/3
ERRORS: 0/3

----
TEST: nums = [-2,1,-3,4,-1,2,1,-5,4]
EXPECTED: 6
GOT: 6
STATUS: OK
MATCH: True

SOLUTION USED:
 class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        if not nums:
            return 0
        max_sum = current_sum = nums[0]
        for num in nums[1:]:
            current_sum = max(num, current_sum + num)
            max_sum = max(max_sum, current_sum)
        return max_sum
----
TEST: nums = [1]
EXPECTED: 1
GOT: 1
STATUS: OK
MATCH: True

SOLUTION USED:
 class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        if not nums:
            return 0
        max_sum = current_sum = nums[0]
        for num in nums[1:]:
            current_sum = max(num, current_sum + num)
            max_sum = max(max_sum, current_sum)
        return max_sum
----
TEST: nums = [5,4,-1,7,8]
EXPECTED: 23
GOT: 23
STATUS: OK
MATCH: True

SOLUTION USED:
 class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        if not nums:
   

## Identified technical bottleneck:

the best test case tht can be generated is upto the limits of the model itself, so we cannot use the code optimising model to generate test cases, we need a model that can test the model to its limits

In [112]:
import json
import re
import typing
import traceback
import copy
import ollama

MODEL_NAME = "llama3"

# =========================
# CLEAN CODE
# =========================
def clean_code(text):
    text = text.replace("```python", "").replace("```", "")

    idx = text.find("class Solution")
    if idx != -1:
        text = text[idx:]

    return text.strip()


# =========================
# VALID CODE CHECK
# =========================
def is_valid_code(code):
    return code and "class Solution" in code


# =========================
# SIGNATURE EXTRACTION
# =========================
import ast

def extract_signature(code):
    try:
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                fn_name = node.name

                params = [arg.arg for arg in node.args.args]

                return fn_name, params

    except Exception as e:
        print("AST parsing failed:", e)

    return None, None



def get_expected_arg_count(tests):
    sample = tests[0]["input_raw"]
    return sample.count("=")


# =========================
# HARD VALIDATION (FIXED + DEBUG)
# =========================
def validate_solution_matches_tests(code, tests):
    fn_name, params = extract_signature(code)

    print("\n=== SIGNATURE DEBUG ===")
    print("Function name:", fn_name)
    print("Params:", params)

    if not fn_name:
        print("❌ No function found")
        return False

    expected_args = get_expected_arg_count(tests)

    # remove self if present
    if params and params[0] == "self":
        params = params[1:]

    actual_args = len(params)

    print("Expected args:", expected_args)
    print("Actual args:", actual_args)

    match = expected_args == actual_args
    print("Signature match:", match)
    print("========================\n")

    return match



# =========================
# INPUT PARSER
# =========================
def parse_inputs(input_raw):
    env = {}

    parts = []
    current = ""
    bracket_level = 0

    for ch in input_raw:
        if ch == ',' and bracket_level == 0:
            parts.append(current.strip())
            current = ""
        else:
            current += ch
            if ch in "[{(":
                bracket_level += 1
            elif ch in "]})":
                bracket_level -= 1

    if current:
        parts.append(current.strip())

    print("Parsed input parts:", parts)

    for part in parts:
        if "=" not in part:
            continue
        exec(part, {}, env)

    print("Parsed values:", list(env.values()))

    return list(env.values())


# =========================
# FUNCTION PICKER
# =========================
def get_function(sol, num_args):
    candidates = []

    print("\n=== FUNCTION PICKER DEBUG ===")

    for name in dir(sol):
        if name.startswith("__"):
            continue

        fn = getattr(sol, name)

        if callable(fn):
            try:
                argcount = fn.__code__.co_argcount - 1
                print(f"Checking {name}: args={argcount}")

                if argcount == num_args:
                    candidates.append((name, fn))
            except:
                continue

    print("Candidates:", [c[0] for c in candidates])

    if not candidates:
        raise Exception("No matching function found")

    for name, fn in candidates:
        if name.lower() in ("solve", "searchrange", "twosum"):
            print("Selected (strong match):", name)
            return fn

    priority = []
    fallback = []

    for name, fn in candidates:
        if name.lower().startswith(("find", "helper", "dfs", "bfs", "util")):
            fallback.append(fn)
        else:
            priority.append(fn)

    chosen = (priority or fallback)[0]
    print("Selected (heuristic):", chosen.__name__)
    print("==============================\n")

    return chosen


# =========================
# RUN SINGLE TEST
# =========================
def run_once(code, test):
    try:
        env = {}

        if "List[" in code and "from typing import" not in code:
            code = "from typing import List\n" + code

        exec(code, env)

        sol = env["Solution"]()

        args = parse_inputs(test["input_raw"])

        print("Args passed to function:", args)

        func = get_function(sol, len(args))

        result = func(*args)

        print("Function result:", result)

        if result is None:
            if len(args) == 1:
                result = args[0]
            else:
                result = args

        return {
            "status": "OK",
            "output": result,
            "error": None,
            "code": code
        }

    except Exception:
        return {
            "status": "ERROR",
            "output": None,
            "error": traceback.format_exc(),
            "code": code
        }


# =========================
# RUN TESTS
# =========================
def run_tests(code, tests):
    results = []

    for i, t in enumerate(tests):
        print(f"\n=== RUNNING TEST {i} ===")

        res = run_once(code, t)

        match = (
            res["status"] == "OK" and
            res["output"] == t["expected"]
        )

        print("Expected:", t["expected"])
        print("Got:", res["output"])
        print("Match:", match)

        results.append({
            "test_id": i,
            "input": t["input_raw"],
            "expected": t["expected"],
            "output": res["output"],
            "status": res["status"],
            "error": res["error"],
            "match": match,
            "solution_used": code
        })

    return results


# =========================
# FULL PIPELINE
# =========================
def full_pipeline(base_solution, description, constraints, tests, max_attempts=3):

    if not is_valid_code(base_solution):
        raise ValueError("Invalid base solution")

    if not validate_solution_matches_tests(base_solution, tests):
        raise ValueError("Base solution does NOT match test signature")

    best_code = base_solution
    best_results = None
    best_score = -1

    for _ in range(max_attempts):

        results = run_tests(best_code, tests)
        passed = sum(r["match"] for r in results)

        if passed > best_score:
            best_score = passed
            best_results = results

        if passed == len(tests):
            break

    # =========================
    # FINAL OUTPUT
    # =========================
    passed = sum(1 for r in best_results if r["match"])
    errors = sum(1 for r in best_results if r["status"] == "ERROR")
    total = len(best_results)

    print(f"\nPASS RATE: {passed}/{total}")
    print(f"ERRORS: {errors}/{total}\n")

    for r in best_results:
        print("----")
        print("TEST:", r["input"])
        print("EXPECTED:", r["expected"])
        print("GOT:", r["output"])
        print("STATUS:", r["status"])
        print("MATCH:", r["match"])

        if not r["match"]:
            print("ERROR:", r["error"])

    print("\nFINAL BEST CODE:\n")
    print(best_code)

    return best_code, best_results


# =========================
# RUN
# =========================
final_code = full_pipeline(
    base_solution=code,
    description=description,
    constraints=constraints,
    tests=tests
)




=== SIGNATURE DEBUG ===
Function name: maxSubArray
Params: ['self', 'nums']
Expected args: 1
Actual args: 1
Signature match: True


=== RUNNING TEST 0 ===
Parsed input parts: ['nums = [-2,1,-3,4,-1,2,1,-5,4]']
Parsed values: [[-2, 1, -3, 4, -1, 2, 1, -5, 4]]
Args passed to function: [[-2, 1, -3, 4, -1, 2, 1, -5, 4]]

=== FUNCTION PICKER DEBUG ===
Checking maxSubArray: args=1
Candidates: ['maxSubArray']
Selected (heuristic): maxSubArray

Function result: 6
Expected: 6
Got: 6
Match: True

=== RUNNING TEST 1 ===
Parsed input parts: ['nums = [1]']
Parsed values: [[1]]
Args passed to function: [[1]]

=== FUNCTION PICKER DEBUG ===
Checking maxSubArray: args=1
Candidates: ['maxSubArray']
Selected (heuristic): maxSubArray

Function result: 1
Expected: 1
Got: 1
Match: True

=== RUNNING TEST 2 ===
Parsed input parts: ['nums = [5,4,-1,7,8]']
Parsed values: [[5, 4, -1, 7, 8]]
Args passed to function: [[5, 4, -1, 7, 8]]

=== FUNCTION PICKER DEBUG ===
Checking maxSubArray: args=1
Candidates: ['maxS

In [113]:
print(final_code[0])

class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        if not nums:
            return 0
        max_sum = current_sum = nums[0]
        for num in nums[1:]:
            current_sum = max(num, current_sum + num)
            max_sum = max(max_sum, current_sum)
        return max_sum


## generating test cases with external tools and testing



1. Boundary testing: checks behavior at minimum, maximum, and edge limits of input constraints
2. Extreme value testing: verifies correctness under very large, very small, or uniform value distributions
3. Pattern testing: validates logic on structured inputs like sorted, reversed, or repeating sequences
4. Random testing: ensures general correctness over unpredictable small-scale inputs
5. Stress testing: evaluates performance and correctness near maximum constraint sizes
6. Deterministic validation: compares exact outputs from candidate and oracle solutions for correctness
7. Failure visibility testing: logs full input-output mismatch details for debugging and analysis
8. Error classification testing: categorizes failures into logic, runtime, or performance issues using rule-based checks

In [114]:
# =========================================================
# INSTALLS
# =========================================================

!pip install hypothesis -q

# =========================================================
# IMPORTS
# =========================================================

import re
import ast
import json
import copy

from typing import *
from hypothesis import given, settings
from hypothesis import strategies as st

# =========================================================
# LOGGER
# =========================================================

def log(stage, msg):

    print(f"\n{'='*70}")
    print(f"[DEBUG - {stage}]")
    print(msg)
    print(f"{'='*70}")


# =========================================================
# CODE EXTRACTOR
# =========================================================

def extract_code_block(raw_text):

    print("\n" + "=" * 70)
    print("[DEBUG - CODE EXTRACTOR]")
    print("RAW MODEL OUTPUT")
    print("=" * 70)

    print(raw_text)

    code = raw_text.strip()

    fenced = re.findall(
        r"```(?:python|Python)?\s*(.*?)```",
        code,
        re.DOTALL
    )

    if fenced:

        code = fenced[0].strip()

    else:

        lines = code.splitlines()

        start = None

        for i, line in enumerate(lines):

            stripped = line.strip()

            if (
                stripped.startswith("def ")
                or stripped.startswith("class ")
                or stripped.startswith("import ")
                or stripped.startswith("from ")
            ):
                start = i
                break

        if start is not None:

            code = "\n".join(lines[start:])

    cleaned_lines = []

    for line in code.splitlines():

        stripped = line.strip()

        if not stripped:

            cleaned_lines.append(line)
            continue

        if (
            stripped.startswith("This ")
            or stripped.startswith("Explanation")
            or stripped.startswith("The function")
            or stripped.startswith("This function")
            or stripped.startswith("Example")
        ):
            break

        cleaned_lines.append(line)

    code = "\n".join(cleaned_lines).strip()

    print("\n" + "=" * 70)
    print("[DEBUG - CODE EXTRACTOR]")
    print("FINAL CLEANED CODE")
    print("=" * 70)

    print(code)

    return code


# =========================================================
# SAFE EXEC
# =========================================================

def safe_exec(code, scope_name="UNKNOWN"):

    log(scope_name, "EXECUTING CODE")

    print(code)

    local_env = {}

    try:

        exec(code, globals(), local_env)

        log(scope_name, "EXEC SUCCESS")

        print("ENV KEYS:")
        print(list(local_env.keys()))

        return local_env

    except Exception as e:

        log(scope_name, f"EXEC FAILED: {e}")

        import traceback
        traceback.print_exc()

        print("\nFAILED CODE:\n")
        print(code)

        raise


# =========================================================
# SANITIZER
# =========================================================

def sanitize_code(code):

    log("SANITIZER", "ORIGINAL CODE")

    print(code)

    lines = code.split("\n")

    fixed = []

    for i, line in enumerate(lines):

        fixed.append(line)

        if re.match(r"^\s*(class|def)\s+.*:\s*$", line):

            indent = len(line) - len(line.lstrip()) + 4

            if i == len(lines) - 1:

                fixed.append(" " * indent + "pass")

    fixed_code = "\n".join(fixed)

    log("SANITIZER", "SANITIZED CODE")

    print(fixed_code)

    return fixed_code


# =========================================================
# SAFE PARSER
# =========================================================

def safe_parse(code):

    log("AST", "PARSING CODE")

    try:

        tree = ast.parse(code)

        log("AST", "PARSE SUCCESS")

        return tree

    except Exception as e:

        log("AST", f"PARSE FAILED: {e}")

        fixed = sanitize_code(code)

        tree = ast.parse(fixed)

        log("AST", "PARSE SUCCESS AFTER SANITIZE")

        return tree


# =========================================================
# SIGNATURE EXTRACTION
# =========================================================

def extract_signature(solution_code):

    log("SIGNATURE", "STARTING EXTRACTION")

    tree = safe_parse(solution_code)

    for node in tree.body:

        # ==========================================
        # CLASS
        # ==========================================

        if isinstance(node, ast.ClassDef):

            print("\nCLASS FOUND:", node.name)

            for item in node.body:

                if isinstance(item, ast.FunctionDef):

                    print("FUNCTION FOUND:", item.name)

                    params = []

                    for arg in item.args.args:

                        if arg.arg == "self":
                            continue

                        annotation = None

                        if arg.annotation:
                            annotation = ast.unparse(arg.annotation)

                        params.append({
                            "name": arg.arg,
                            "type": annotation
                        })

                    signature = {
                        "type": "class",
                        "method": item.name,
                        "params": params
                    }

                    print(signature)

                    return signature

        # ==========================================
        # FUNCTION
        # ==========================================

        if isinstance(node, ast.FunctionDef):

            params = []

            for arg in node.args.args:

                annotation = None

                if arg.annotation:
                    annotation = ast.unparse(arg.annotation)

                params.append({
                    "name": arg.arg,
                    "type": annotation
                })

            signature = {
                "type": "function",
                "method": node.name,
                "params": params
            }

            print(signature)

            return signature

    raise Exception("NO FUNCTION FOUND")


# =========================================================
# LOAD SOLUTION
# =========================================================

# =========================================================
# LOAD SOLUTION
# =========================================================

def load_solution(solution_code, signature):

    log("LOAD SOLUTION", "STARTING")

    solution_code = sanitize_code(solution_code)

    local_env = safe_exec(
        solution_code,
        "USER SOLUTION"
    )

    # ==========================================
    # CLASS SOLUTION
    # ==========================================

    if "Solution" in local_env:

        obj = local_env["Solution"]()

        method_name = signature["method"]

        print("SELECTED METHOD:", method_name)

        return getattr(obj, method_name)

    # ==========================================
    # FUNCTION SOLUTION
    # ==========================================

    funcs = [
        v for v in local_env.values()
        if callable(v)
    ]

    if not funcs:
        raise Exception("NO CALLABLE FOUND")

    return funcs[0]


# =========================================================
# CONSTRAINT PARSER
# =========================================================

def parse_constraints(constraints):

    log("CONSTRAINTS", "PARSING")

    parsed = {}

    # ==========================================
    # LENGTH
    # ==========================================

    length_matches = re.findall(
        r'(\d+)\s*<=\s*(\w+)\.length\s*<=\s*(10\^\d+|\d+)',
        constraints
    )

    for match in length_matches:

        min_len = eval(match[0].replace("^", "**"))
        var_name = match[1]
        max_len = eval(match[2].replace("^", "**"))

        parsed[var_name] = {
            "min_size": min_len,
            "max_size": min(max_len, 50)
        }

    # ==========================================
    # ARRAY RANGE
    # ==========================================

    value_matches = re.findall(
        r'(-?(?:10\^\d+|\d+))\s*<=\s*(\w+)\[i\]\s*<=\s*(-?(?:10\^\d+|\d+))',
        constraints
    )

    for match in value_matches:

        min_val = eval(match[0].replace("^", "**"))
        var_name = match[1]
        max_val = eval(match[2].replace("^", "**"))

        if var_name not in parsed:
            parsed[var_name] = {}

        parsed[var_name]["element_min"] = min_val
        parsed[var_name]["element_max"] = max_val

    print(parsed)

    return parsed


# =========================================================
# STRATEGY FACTORY
# =========================================================

def strategy_from_type(type_name, constraints):

    print("\nBUILDING STRATEGY:", type_name)

    # ==========================================
    # INT
    # ==========================================

    if type_name == "int":

        return st.integers(
            min_value=constraints.get("min", -100),
            max_value=constraints.get("max", 100)
        )

    # ==========================================
    # LIST INT
    # ==========================================

    if type_name == "List[int]":

        return st.lists(
            st.integers(
                min_value=constraints.get(
                    "element_min",
                    -100
                ),
                max_value=constraints.get(
                    "element_max",
                    100
                )
            ),
            min_size=constraints.get(
                "min_size",
                0
            ),
            max_size=min(
                constraints.get(
                    "max_size",
                    20
                ),
                50
            )
        )

    # ==========================================
    # STRING
    # ==========================================

    if type_name == "str":

        return st.text(
            min_size=0,
            max_size=20
        )

    # ==========================================
    # BOOL
    # ==========================================

    if type_name == "bool":

        return st.booleans()

    # ==========================================
    # FALLBACK
    # ==========================================

    return st.none()


# =========================================================
# BUILD STRATEGIES
# =========================================================

def build_strategies(signature, constraints):

    log("BUILD STRATEGIES", "STARTING")

    parsed_constraints = parse_constraints(
        constraints
    )

    strategies = {}

    for param in signature["params"]:

        name = param["name"]
        typ = param["type"]

        strategies[name] = strategy_from_type(
            typ,
            parsed_constraints.get(name, {})
        )

    print(strategies)

    return strategies


# =========================================================
# BUILD ORACLE FROM EXISTING SOLUTION
# =========================================================

def build_oracle_from_solution(final_code):

    print("\n" + "="*70)
    print("[DEBUG - ORACLE]")
    print("USING EXISTING SOLUTION")
    print("="*70)

    oracle_code = final_code[0]

    print("\nORACLE CODE:\n")
    print(oracle_code)

    return oracle_code


# =========================================================
# LOAD ORACLE
# =========================================================

def load_oracle(oracle_code, signature):

    log("LOAD ORACLE", "STARTING")

    oracle_env = safe_exec(
        oracle_code,
        "ORACLE"
    )

    print("\nORACLE ENV:")
    print(oracle_env)

    if "Solution" in oracle_env:

        print("\nFOUND Solution CLASS")

        oracle_instance = oracle_env["Solution"]()

        methods = [
            m for m in dir(oracle_instance)
            if not m.startswith("__")
        ]

        print("\nMETHODS:")
        print(methods)

        oracle_func = getattr(
            oracle_instance,
            signature["method"]
        )

        print("\nSELECTED METHOD:")
        print(signature["method"])

        return oracle_func

    elif "oracle" in oracle_env:

        print("\nFOUND oracle FUNCTION")

        return oracle_env["oracle"]

    else:

        raise Exception(
            "FAILED TO LOAD ORACLE"
        )


# =========================================================
# VALIDATE ORACLE
# =========================================================

# =========================================================
# VALIDATE ORACLE
# =========================================================

def validate_oracle(
    oracle_func,
    solution_func,
    tests,
    signature
):

    print("\n" + "=" * 70)
    print("[DEBUG - VALIDATE ORACLE]")
    print("STARTING")
    print("=" * 70)

    for idx, test in enumerate(tests):

        print(f"\nTEST #{idx}")

        print("\nRAW TEST:")
        print(test)

        kwargs = {}

        assignments = []

        raw_input = test["input_raw"]

        print("\nRAW INPUT:")
        print(raw_input)

        # ======================================
        # FIXED PARSING
        # ======================================

        pattern = r'(\w+)\s*=\s*(\[.*?\]|".*?"|\'.*?\'|-?\d+|True|False)'

        matches = re.findall(pattern, raw_input)

        print("\nMATCHES:")
        print(matches)

        for name, value in matches:

            assignments.append(
                (name.strip(), value.strip())
            )

        print("\nASSIGNMENTS:")
        print(assignments)

        for name, value in assignments:

            print(f"\nNAME: {name}")
            print(f"VALUE: {value}")

            kwargs[name] = ast.literal_eval(value)

        print("\nPARSED KWARGS:")
        print(kwargs)

        expected = test["expected"]

        print("\nEXPECTED:")
        print(expected)

        # ======================================
        # ORACLE
        # ======================================

        oracle_kwargs = copy.deepcopy(kwargs)

        oracle_output = oracle_func(**oracle_kwargs)

        # inplace support
        if oracle_output is None:

            param_name = signature["params"][0]["name"]

            oracle_output = oracle_kwargs[param_name]

        print("\nORACLE OUTPUT:")
        print(oracle_output)

        # ======================================
        # SOLUTION
        # ======================================

        solution_kwargs = copy.deepcopy(kwargs)

        solution_output = solution_func(
            **solution_kwargs
        )

        # inplace support
        if solution_output is None:

            param_name = signature["params"][0]["name"]

            solution_output = solution_kwargs[param_name]

        print("\nSOLUTION OUTPUT:")
        print(solution_output)

        # ======================================
        # ASSERTIONS
        # ======================================

        assert oracle_output == expected, f"""

ORACLE FAILED EXAMPLE

INPUT:
{kwargs}

EXPECTED:
{expected}

ORACLE:
{oracle_output}
"""

        assert solution_output == expected, f"""

SOLUTION FAILED EXAMPLE

INPUT:
{kwargs}

EXPECTED:
{expected}

SOLUTION:
{solution_output}
"""

    print("\n" + "=" * 70)
    print("[DEBUG - VALIDATE ORACLE]")
    print("VALIDATION PASSED")
    print("=" * 70)


# =========================================================
# MAIN PIPELINE
# =========================================================

def generate_tests(structured_data, tests):

    log("PIPELINE", "STARTING")

    description = structured_data["description"]

    constraints = structured_data["constraints"]

    solution = structured_data["solution"]

    print("\nDESCRIPTION:\n")
    print(description)

    print("\nCONSTRAINTS:\n")
    print(constraints)

    print("\nSOLUTION:\n")
    print(solution)

    # ==========================================
    # SIGNATURE
    # ==========================================

    signature = extract_signature(
        solution
    )

    # ==========================================
    # LOAD SOLUTION
    # ==========================================


    solution_func = load_solution(
      solution,
      signature
)

    # ==========================================
    # STRATEGIES
    # ==========================================

    strategies = build_strategies(
        signature,
        constraints
    )

    # ==========================================
    # ORACLE FROM EXISTING SOLUTION
    # ==========================================

    oracle_code = build_oracle_from_solution(
        final_code
    )

    oracle_func = load_oracle(
        oracle_code,
        signature
    )

    # ==========================================
    # VALIDATE
    # ==========================================

    validate_oracle(
        oracle_func,
        solution_func,
        tests,
        signature
    )

    # ==========================================
    # PROPERTY TEST
    # ==========================================

    log("HYPOTHESIS", "STARTING")

    @settings(max_examples=20)
    @given(**strategies)
    def property_test(**kwargs):

        print("\nGENERATED INPUT:")
        print(kwargs)

        expected_kwargs = copy.deepcopy(kwargs)
        actual_kwargs = copy.deepcopy(kwargs)

        expected = oracle_func(
            **expected_kwargs
        )

        if expected is None:

            param_name = signature["params"][0]["name"]

            expected = expected_kwargs[param_name]

        actual = solution_func(
            **actual_kwargs
        )

        if actual is None:

            param_name = signature["params"][0]["name"]

            actual = actual_kwargs[param_name]

        print("EXPECTED:", expected)
        print("ACTUAL:", actual)

        assert actual == expected

    property_test()

    log("PIPELINE", "ALL TESTS PASSED")


# =========================================================
# IMPORTANT
# =========================================================

structured_data["solution"] = final_code[0]

# =========================================================
# RUN
# =========================================================

generate_tests(
    structured_data,
    tests
)


[DEBUG - PIPELINE]
STARTING

DESCRIPTION:

Given an integer array nums, find the subarray with the largest sum, and return its sum.

CONSTRAINTS:

1 <= nums.length <= 10^5
-10^4 <= nums[i] <= 10^4

SOLUTION:

class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        if not nums:
            return 0
        max_sum = current_sum = nums[0]
        for num in nums[1:]:
            current_sum = max(num, current_sum + num)
            max_sum = max(max_sum, current_sum)
        return max_sum

[DEBUG - SIGNATURE]
STARTING EXTRACTION

[DEBUG - AST]
PARSING CODE

[DEBUG - AST]
PARSE SUCCESS

CLASS FOUND: Solution
FUNCTION FOUND: maxSubArray
{'type': 'class', 'method': 'maxSubArray', 'params': [{'name': 'nums', 'type': 'List[int]'}]}

[DEBUG - LOAD SOLUTION]
STARTING

[DEBUG - SANITIZER]
ORIGINAL CODE
class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        if not nums:
            return 0
        max_sum = current_sum = nums[0]
        for num in nums[1:

In [115]:
def llm_compare_solutions(code, final_code, description):

    optimized_code = final_code[0]

    prompt = f"""
You are a senior programming tutor and code reviewer.

Your task:
Compare BASE and OPTIMIZED solutions line-by-line and explain improvements.

You must:
- Explain differences in simple but precise terms
- Identify correctness improvements
- Identify performance/complexity changes (if any)
- Say if optimization is REAL or just REFACTORING
- Be strict: do NOT assume improvement unless it exists

========================
PROBLEM DESCRIPTION
========================
{description}

========================
BASE SOLUTION
========================
{code}

========================
OPTIMIZED SOLUTION
========================
{final_code[0]}

========================
OUTPUT FORMAT
========================

1. OVERALL SUMMARY
- explain how you planned to solve the problem
- What both solutions do
- Whether optimized is truly better or not

2. LINE-BY-LINE DIFFERENCE
For each meaningful change:
- What changed
- Why it changed
- Impact on correctness or efficiency

3. LOGIC ANALYSIS
- Are both logically equivalent?
- Any bug fixes introduced?

4. COMPLEXITY COMPARISON
- Time complexity (base vs optimized)
- Space complexity

5. FINAL VERDICT
- "True Optimization" OR "Refactor Only" OR "Regression Risk"
"""

    res = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={"temperature": 0}
    )

    return res["message"]["content"]

In [116]:
report = llm_compare_solutions(
    code,
    final_code,
    description
)

print(report)

print('---------------------------- base solution ----------------------------------')
print(code)

print('---------------------------- optimized solution -----------------------------')
print(final_code[0])

**OVERALL SUMMARY**

The problem is to find the subarray with the largest sum in a given integer array and return its sum. Both solutions, BASE and OPTIMIZED, aim to achieve this goal.

Upon reviewing both solutions, I can conclude that they are logically equivalent and do essentially the same thing: iterate through the input array, keeping track of the maximum sum found so far and the current sum. The optimized solution does not introduce any bug fixes or significant changes in logic.

**LINE-BY-LINE DIFFERENCE**

1. No differences between BASE and OPTIMIZED solutions. They are identical line-by-line.
2. Since there are no meaningful changes, I will not provide a detailed analysis of each line.

**LOGIC ANALYSIS**

Both BASE and OPTIMIZED solutions have the same logic: they iterate through the input array, updating the maximum sum found so far and the current sum. The optimized solution does not introduce any bug fixes or significant changes in logic.

**COMPLEXITY COMPARISON**

1. Ti